In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents: 
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [10]:
import os
print("Is key loaded?:", os.environ.get("GROQ_API_KEY") is not None)

Is key loaded?: True


In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [19]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [22]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index = index,
    llm_client = openai_client,
)

In [23]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"Yes, it's okay to join the course late. According to the context, if you've just discovered the course, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted."

In [24]:
assistant.total_cost()

0.0004458

In [26]:
doc_id = rec['document']
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index = index,
    llm_client = openai_client,
)

In [ ]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"Yes, it's okay to join the course late. According to the context, if you've just discovered the course, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted."

In [32]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result
def generate_rag_answer(rec):
    question = rec['question']
    doc_id = rec['document']
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc['answer']

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result
answer_record = generate_rag_answer(ground_truth[0])
answer_record
assistant.reset_usage()
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


In [33]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)